In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 56. Week 38 — Software engineering for research

## 学習目標

- exploratory cellからpure APIとI/O adapterを分離できる
- configuration、type、error、logのcontractを書ける
- example/unit/property/integration testを役割別に設計できる
- content-addressed experiment runを再生成できる

## 前提知識

- Python package、dataclass、exception
- Week 16のresearch software
- B9 feature/model API

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 56


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask
assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("locked outer rows present: False")

fixture rows: 256
inner train / validation: 192 64
locked outer rows present: False


In [4]:
numeric_preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric_features = numeric_preprocessor.transform(fixture.numeric_features)
numeric_model = qt.fit_sparse_ridge(
    numeric_features[train_mask], fixture.targets[train_mask], ridge=1.0
)
numeric_validation_prediction = numeric_model.predict(numeric_features[validation_mask])
numeric_metrics = qt.regression_error_table(
    fixture.targets[validation_mask],
    numeric_validation_prediction,
    np.asarray(fixture.entity_ids)[validation_mask],
)
print("numeric validation metrics:", numeric_metrics)

numeric validation metrics: {'mae': 0.06965156975112882, 'median_absolute_error': 0.02798038529997949, 'rmse': 0.15698922708171414, 'company_macro_mae': 0.05985240327713495}


## 1. Package boundary

`quant_textbook`はpure numerical/data contractを`src/`へ置き、Notebookは設定・可視化・解釈を担当する。raw SEC network/cache adapter、feature transform、model、metric、artifact hashを一つのfunctionへ混ぜない。

| layer | input | output | failure |
|---|---|---|---|
| data adapter | path/manifest | validated records | missing/hash/schema error |
| features | records + training mask | transform + matrix | leakage/shape/nonfinite |
| model | matrix + config | immutable parameters | rank/convergence/budget |
| evaluation | target/pred/entity | metric table | mismatch/nonfinite |
| registry | run metadata/hash | append-only state | duplicate/tamper |

In [5]:
import hashlib

fixture_digest = "6487c20568fbbbb18326bcde49d985b42ddd5072f321e33e4b2b8154a7293295"
prediction_digest = hashlib.sha256(
    numeric_validation_prediction.astype("<f8", copy=False).tobytes()
).hexdigest()
config = {
    "family": "numeric_ridge",
    "ridge": 1.0,
    "training_rows": int(train_mask.sum()),
    "validation_rows": int(validation_mask.sum()),
    "outer_access": "unopened",
}
run = qt.build_experiment_run(
    experiment_name="b10-teaching-reproduction",
    candidate_name="numeric-ridge",
    stage="development",
    config=config,
    data_sha256=fixture_digest,
    code_revision="notebook-56-generated-source",
    metrics=numeric_metrics,
    artifact_sha256={"validation_prediction": prediction_digest},
)
repeated = qt.build_experiment_run(
    experiment_name="b10-teaching-reproduction",
    candidate_name="numeric-ridge",
    stage="development",
    config=config,
    data_sha256=fixture_digest,
    code_revision="notebook-56-generated-source",
    metrics=numeric_metrics,
    artifact_sha256={"validation_prediction": prediction_digest},
)
assert run == repeated
display(pd.DataFrame([{"run_id": run.run_id, "config_sha256": run.config_sha256, "data_sha256": run.data_sha256, "prediction_sha256": prediction_digest}]))

,run_id,config_sha256,data_sha256,prediction_sha256
0,6c842fe3307764b63c1cb63f,8d3a1706936e28c0e5d9a3da4a98d3c1d15ee01ac08a03...,6487c20568fbbbb18326bcde49d985b42ddd5072f321e3...,0cd72c914c8cd79c2ff190492e4130d965b9ddce54c50a...


## 2. Test portfolioとfailure semantics

unit testは局所契約、property-style testは多入力のinvariant、integration testはlayer間契約、Notebook executionはreader-facing evidenceを検査する。例が一つ通っただけでpropertyを証明しない。invalid inputをwarningだけで続行せず、例外type/messageをcontractにする。

In [6]:
test_portfolio = pd.DataFrame(
    [
        {"test_type": "known-answer unit", "examples": 14, "target": "gradient/hash/metric"},
        {"test_type": "edge case", "examples": 12, "target": "empty/nonfinite/schema/time"},
        {"test_type": "property-style", "examples": 8, "target": "determinism/invariance/coverage"},
        {"test_type": "integration", "examples": 6, "target": "builder/notebook/book"},
    ]
)
fig = go.Figure()
fig.add_bar(x=test_portfolio["test_type"], y=test_portfolio["examples"])
fig.update_layout(title="Test roles are complementary", yaxis_title="Illustrative checks", template="plotly_white")
fig.show()

try:
    qt.fit_sparse_ridge(np.ones((2, 1)), np.array([1.0, np.nan]), ridge=1.0)
except ValueError as error:
    failure_message = str(error)
else:
    raise AssertionError("invalid target must fail closed")
print("expected failure:", failure_message)

expected failure: target must be finite with one value per row


## 3. Logging、configuration、environment

logはhuman proseでなくevent name、run ID、stage、input/output hash、duration、statusをstructured fieldにする。secret、contact、raw filing本文をlogへ入れない。configurationはversioned schemaとして保存し、environment lockはdependencyを宣言するが、共有`.venv`に偶然あるpackageを依存contractにはしない。

## 4. 失敗モード

- Notebook global stateをlibrary APIが読む
- broad `except Exception`でpartial artifactをsuccessにする
- configurationを後から上書きして同じrun IDを使う
- test数だけをquality metricにする
- logへsecret/raw documentを出す

## 5. 段階別演習

### 基礎

1. data/features/model/evaluationのinterfaceを書け。
2. invalid targetがfail-closedになるtestを書け。

### 標準

3. config一項変更でrun IDが変わるpropertyを確認せよ。
4. clean processで同じprediction hashを再生成せよ。

### 研究

5. CI matrixにPython/OS/BLAS差を入れる費用と価値を評価せよ。

## 6. Exit Criteria

- [ ] package layerとNotebook責務を分離した
- [ ] config/data/code/output hashをrunへ結んだ
- [ ] unit/property/integration testを区別した
- [ ] invalid inputをfail-closedにした
- [ ] undeclared dependencyとsecretをartifactへ入れていない

## 7. 出典

- [Python Packaging User Guide](https://packaging.python.org/en/latest/)
- [Python logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [pytest documentation](https://docs.pytest.org/)
- [Semantic Versioning 2.0.0](https://semver.org/)